# Clase 06 — Transformación de Datos: Registro de Clientes

**Asignatura:** Ingeniería de Datos  
**Docente:** Ing. Sergio Orozco

En esta práctica aplicamos las técnicas de **validación, limpieza y transformación**  
sobre un dataset de clientes con problemas reales de calidad de datos.

El foco está en la **validación de formato y reglas de negocio** antes de aceptar  
un registro en el sistema de destino.

| Sección | Tema |
|---------|------|
| **1** | Instalación e importación de librerías |
| **2** | Carga y diagnóstico del dataset crudo |
| **3** | Validaciones de formato y reglas de negocio |
| **4** | Consolidación: registros válidos e inválidos |
| **5** | Normalización |
| **6** | Derivación de columnas calculadas |
| **7** | Resumen y estadísticas |
| **8** | Guardado de resultados |

**Tecnologías:** `pandas`, `numpy`, `re`

## 1. Instalación e Importación de Librerías

In [1]:
%pip install pandas numpy openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ── Rutas de trabajo ──────────────────────────────────────────────────────────
DIR_INPUT  = Path("datos/input")
DIR_OUTPUT = Path("datos/output")
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Librerías importadas correctamente.")
print(f"pandas  versión: {pd.__version__}")
print(f"numpy   versión: {np.__version__}")

Librerías importadas correctamente.
pandas  versión: 3.0.2
numpy   versión: 2.4.4


## 2. Carga y Diagnóstico del Dataset Crudo

Cargamos el archivo `clientes_crudos.csv` y realizamos un **diagnóstico inicial**  
para entender qué problemas de calidad contiene el dataset antes de procesarlo.

> El dataset simula datos de clientes tal como llegarían desde un sistema externo,  
> con errores de formato, campos faltantes e inconsistencias.

In [3]:
# Cargar el dataset de clientes crudos
# dtype=str para que pandas no intente inferir tipos y preserve los valores tal cual
df_crudo = pd.read_csv(
    DIR_INPUT / "clientes_crudos.csv",
    dtype=str,
    keep_default_na=False,  # vacíos como string vacío, no como NaN
)

# Reemplazar strings vacíos por NaN para facilitar el análisis de nulos
df_crudo = df_crudo.replace("", np.nan)

print("=== PRIMERAS FILAS ===")
df_crudo

=== PRIMERAS FILAS ===


,id_cliente,nombres,apellidos,numero_documento,fecha_nacimiento,email,nro_telefono,direccion,sexo,fecha_alta,fecha_baja
0,1,Juan,Pérez,12345678,1990-05-15,juan.perez@gmail.com,+5491123456789,Av. Corrientes 1234 CABA,M,2020-01-10,NaN
1,2,María,García,23456789,1985-03-22,maria.garcia@hotmail.com,+5491187654321,Calle Falsa 123 Rosario,F,2019-06-15,NaN
2,3,Carlos,López,34567890,1978-11-08,carlos.lopez@yahoo.com,+5491112345678,Belgrano 456 Córdoba,M,2021-03-01,NaN
3,4,Ana,Martínez,45678901,1995-07-30,ana.martinez@outlook.com,+5491198765432,San Martín 789 Mendoza,F,2022-08-20,NaN
4,5,Luis,Fernández,56789012,2000-01-25,luis.fernandez@gmail.com,+5491134567890,Rivadavia 321 La Plata,M,2023-02-14,NaN
5,6,Laura,González,67890123,1988-09-12,laura.gonzalez@gmail.com,+5491156789012,Florida 100 CABA,F,2018-11-30,NaN
6,7,Pedro,Rodríguez,78901234,1975-04-03,pedro.rodriguez@hotmail.com,+5491178901234,Mitre 555 Tucumán,M,2017-05-22,NaN
7,8,Sofía,Ramírez,89012345,1992-12-19,sofia.ramirez@gmail.com,+5491190123456,Sarmiento 678 Salta,F,2024-01-05,NaN
8,9,Diego,Torres,90123456,1983-06-07,diego.torres@yahoo.com,+5491112367890,Alberdi 234 Mar del Plata,M,2020-09-18,NaN
9,10,Valentina,Moreno,01234567,2003-08-14,valentina.moreno@gmail.com,+5491145678901,Lavalle 890 CABA,F,2025-03-10,NaN


In [4]:
# ── Diagnóstico inicial ────────────────────────────────────────────────────────
print("=== DIMENSIONES ===")
print(f"Filas: {len(df_crudo)}  |  Columnas: {len(df_crudo.columns)}")

print("\n=== VALORES NULOS POR COLUMNA ===")
print(df_crudo.isnull().sum())

print("\n=== VALORES ÚNICOS EN 'sexo' ===")
print(df_crudo["sexo"].value_counts(dropna=False))

print("\n=== REGISTROS CON fecha_baja ===")
print(df_crudo["fecha_baja"].notna().sum())

print("\n=== FILAS COMPLETAMENTE DUPLICADAS ===")
print(f"Duplicados: {df_crudo.duplicated().sum()}")

=== DIMENSIONES ===
Filas: 17  |  Columnas: 11

=== VALORES NULOS POR COLUMNA ===
id_cliente           0
nombres              0
apellidos            0
numero_documento     1
fecha_nacimiento     1
email                1
nro_telefono         1
direccion            0
sexo                 0
fecha_alta           0
fecha_baja          15
dtype: int64

=== VALORES ÚNICOS EN 'sexo' ===
sexo
M    8
F    8
X    1
Name: count, dtype: int64

=== REGISTROS CON fecha_baja ===
2

=== FILAS COMPLETAMENTE DUPLICADAS ===
Duplicados: 0


### Problemas detectados en el dataset

| # | Problema | Columna | Tipo |
|---|----------|---------|------|
| 1 | `fecha_nacimiento` vacía | `fecha_nacimiento` | Campo obligatorio ausente |
| 2 | Sin email NI teléfono | `email`, `nro_telefono` | Sin medio de contacto |
| 3 | `numero_documento` vacío | `numero_documento` | Campo obligatorio ausente |
| 4 | Email sin formato válido | `email` | Error de formato |
| 5 | Teléfono sin formato válido | `nro_telefono` | Error de formato |

> Los casos 1 y 2 ya tienen `fecha_baja` asignada en la fuente, indicando que  
> el cliente fue dado de baja previamente por esos mismos motivos.

## 3. Validaciones de Formato y Reglas de Negocio

Para cada regla de validación:
1. Definimos el criterio
2. Aplicamos una función o expresión regular al dataset
3. Marcamos con `True` (pasa) o `False` (no pasa) cada registro

Al final consolidaremos todos los resultados en una columna `observaciones`.

### 3.1 Patrones de Validación (Expresiones Regulares)

Definimos los patrones regex que usaremos a lo largo de toda la validación.

| Campo | Patrón | Descripción |
|-------|--------|-------------|
| `email` | `^[\w\.\+\-]+@[\w\-]+(\.[a-zA-Z]{2,})+$` | Usuario @ dominio . extensión |
| `numero_documento` | `^\d{7,8}$` | 7 u 8 dígitos numéricos |
| `nro_telefono` | `^\+?\d{8,15}$` | Opcional `+`, 8 a 15 dígitos |

In [5]:
# ── Patrones de validación ─────────────────────────────────────────────────────
RE_EMAIL   = re.compile(r'^[\w\.\+\-]+@[\w\-]+(\.[a-zA-Z]{2,})+$')
RE_DNI     = re.compile(r'^\d{7,8}$')
RE_TELEFONO = re.compile(r'^\+?\d{8,15}$')
RE_SOLO_LETRAS = re.compile(r'^[a-záéíóúüñA-ZÁÉÍÓÚÜÑ\s\-\']+$')  # Solo letras y guiones

SEXOS_VALIDOS = {"M", "F", "X"}

# Función auxiliar: aplica un regex, devuelve False si el valor es nulo
def match_regex(patron, valor):
    """Retorna True si el valor no es nulo y coincide con el patrón."""
    if pd.isna(valor):
        return False
    return bool(patron.match(str(valor).strip()))

print("Patrones definidos correctamente.")

Patrones definidos correctamente.


### 3.2 Validación de Nombres y Apellidos

**Reglas:**
- No pueden estar vacíos
- No pueden contener números
- Solo deben contener letras, espacios y guiones

In [6]:
df_val = df_crudo.copy()

# ── Validación: nombres ────────────────────────────────────────────────────────
df_val["ok_nombres"] = df_val["nombres"].apply(
    lambda v: match_regex(RE_SOLO_LETRAS, v)
)

# ── Validación: apellidos ──────────────────────────────────────────────────────
df_val["ok_apellidos"] = df_val["apellidos"].apply(
    lambda v: match_regex(RE_SOLO_LETRAS, v)
)

print("=== NOMBRES INVÁLIDOS ===")
print(df_val[~df_val["ok_nombres"]][["id_cliente", "nombres", "apellidos"]])

print("\n=== APELLIDOS INVÁLIDOS ===")
print(df_val[~df_val["ok_apellidos"]][["id_cliente", "nombres", "apellidos"]])

=== NOMBRES INVÁLIDOS ===
Empty DataFrame
Columns: [id_cliente, nombres, apellidos]
Index: []

=== APELLIDOS INVÁLIDOS ===
Empty DataFrame
Columns: [id_cliente, nombres, apellidos]
Index: []


### 3.3 Validación de Número de Documento (DNI)

**Regla:** debe contener entre 7 y 8 dígitos numéricos, sin letras ni símbolos.

In [7]:
# ── Validación: número de documento ───────────────────────────────────────────
df_val["ok_dni"] = df_val["numero_documento"].apply(
    lambda v: match_regex(RE_DNI, v)
)

print("=== DOCUMENTOS INVÁLIDOS O AUSENTES ===")
problemas_dni = df_val[~df_val["ok_dni"]][["id_cliente", "nombres", "apellidos", "numero_documento"]]
print(problemas_dni)

=== DOCUMENTOS INVÁLIDOS O AUSENTES ===
   id_cliente nombres apellidos numero_documento
14         15  Andrés    Castro              NaN


### 3.4 Validación de Email

**Regla:** si está presente, debe respetar el formato `usuario@dominio.extension`.  
Un email vacío no es error por sí solo (se gestiona en la regla de contacto).

In [8]:
# ── Validación: formato de email ───────────────────────────────────────────────
# Un email nulo pasa esta validación (será evaluado en la regla de contacto)
df_val["ok_email_formato"] = df_val["email"].apply(
    lambda v: True if pd.isna(v) else match_regex(RE_EMAIL, v)
)

print("=== EMAILS CON FORMATO INVÁLIDO ===")
problemas_email = df_val[~df_val["ok_email_formato"]][["id_cliente", "nombres", "apellidos", "email"]]
print(problemas_email)

=== EMAILS CON FORMATO INVÁLIDO ===
   id_cliente  nombres apellidos                   email
15         16  Natalia     Silva  natalia.silva.noarroba


### 3.5 Validación de Número de Teléfono

**Regla:** si está presente, debe contener entre 8 y 15 dígitos, con `+` opcional al inicio.  
Un teléfono vacío no es error por sí solo (se gestiona en la regla de contacto).

In [ ]:
# ── Validación: formato de teléfono ───────────────────────────────────────────
df_val["ok_telefono_formato"] = df_val["nro_telefono"].apply(
    lambda v: True if pd.isna(v) else match_regex(RE_TELEFONO, v)
)

print("=== TELÉFONOS CON FORMATO INVÁLIDO ===")
problemas_tel = df_val[~df_val["ok_telefono_formato"]][["id_cliente", "nombres", "apellidos", "nro_telefono"]]
print(problemas_tel)

### 3.6 Validación de Medio de Contacto

**Regla de negocio:** todo cliente activo debe tener **al menos un medio de contacto** válido:  
un `email` con formato correcto, un `nro_telefono` con formato correcto, o ambos.

> Sin contacto no es posible comunicarse con el cliente,  
> por lo que el registro no puede ser admitido en el sistema.

In [ ]:
# ── Validación: al menos un medio de contacto válido ──────────────────────────
# Tiene email válido si no es nulo Y pasa el formato
tiene_email_valido    = df_val["email"].notna() & df_val["ok_email_formato"]
tiene_telefono_valido = df_val["nro_telefono"].notna() & df_val["ok_telefono_formato"]

df_val["ok_contacto"] = tiene_email_valido | tiene_telefono_valido

print("=== REGISTROS SIN MEDIO DE CONTACTO VÁLIDO ===")
sin_contacto = df_val[~df_val["ok_contacto"]][["id_cliente", "nombres", "apellidos", "email", "nro_telefono"]]
print(sin_contacto)

### 3.7 Validación de Fechas

**Reglas:**
- `fecha_nacimiento`: obligatoria, debe ser una fecha válida
- `fecha_alta`: obligatoria, debe ser una fecha válida
- `fecha_baja`: opcional, pero si existe debe ser una fecha válida y posterior a `fecha_alta`

Usamos `pd.to_datetime(..., errors='coerce')` que convierte valores inválidos en `NaT`.

In [ ]:
# ── Parseo de fechas ───────────────────────────────────────────────────────────
df_val["fecha_nacimiento_dt"] = pd.to_datetime(df_val["fecha_nacimiento"], errors="coerce")
df_val["fecha_alta_dt"]       = pd.to_datetime(df_val["fecha_alta"],       errors="coerce")
df_val["fecha_baja_dt"]       = pd.to_datetime(df_val["fecha_baja"],       errors="coerce")

# ── Validación: fecha_nacimiento presente y válida ─────────────────────────────
df_val["ok_fecha_nac"] = df_val["fecha_nacimiento_dt"].notna()

# ── Validación: fecha_alta presente y válida ───────────────────────────────────
df_val["ok_fecha_alta"] = df_val["fecha_alta_dt"].notna()

# ── Validación: fecha_baja válida SI existe (NaN si está vacía está permitido) ─
df_val["ok_fecha_baja"] = (
    df_val["fecha_baja"].isna()  # Puede no tener fecha_baja (cliente activo)
    | df_val["fecha_baja_dt"].notna()  # O si la tiene, debe ser válida
)

# ── Validación: fecha_baja debe ser posterior a fecha_alta ─────────────────────
# Solo aplica a filas donde ambas fechas existen y son válidas
ambas_fechas = df_val["fecha_alta_dt"].notna() & df_val["fecha_baja_dt"].notna()
baja_posterior = df_val["fecha_baja_dt"] >= df_val["fecha_alta_dt"]
df_val["ok_baja_posterior"] = ~ambas_fechas | baja_posterior  # Si no aplica, True

print("=== FECHAS DE NACIMIENTO INVÁLIDAS O AUSENTES ===")
print(df_val[~df_val["ok_fecha_nac"]][["id_cliente", "nombres", "apellidos", "fecha_nacimiento"]])

print("\n=== FECHAS DE ALTA INVÁLIDAS O AUSENTES ===")
print(df_val[~df_val["ok_fecha_alta"]][["id_cliente", "nombres", "apellidos", "fecha_alta"]])

print("\n=== FECHAS DE BAJA INVÁLIDAS (cuando existen) ===")
print(df_val[~df_val["ok_fecha_baja"]][["id_cliente", "nombres", "apellidos", "fecha_baja"]])

### 3.8 Validación de Sexo

**Regla:** el campo `sexo` debe contener únicamente los valores aceptados: `M`, `F` o `X`.  
No se permiten valores nulos ni variantes distintas.

In [ ]:
# ── Validación: campo sexo ─────────────────────────────────────────────────────
df_val["ok_sexo"] = df_val["sexo"].apply(
    lambda v: str(v).strip().upper() in SEXOS_VALIDOS if pd.notna(v) else False
)

print("=== VALORES DE SEXO INVÁLIDOS ===")
print(df_val[~df_val["ok_sexo"]][["id_cliente", "nombres", "apellidos", "sexo"]])

print(f"\nValores únicos encontrados: {df_val['sexo'].unique()}")

## 4. Consolidación: Registros Válidos e Inválidos

Reunimos todas las validaciones en una sola columna `observaciones` que lista  
cada problema encontrado por registro.

Luego separamos el dataset en dos subconjuntos:
- **`df_validos`**: registros que pasaron todas las validaciones
- **`df_rechazados`**: registros con al menos un problema, con la descripción de cada error

In [ ]:
# ── Construcción de la columna 'observaciones' ─────────────────────────────────
# Para cada fila, acumulamos en una lista los mensajes de error correspondientes

def construir_observaciones(row):
    errores = []
    if not row["ok_nombres"]:
        errores.append("Nombre inválido o vacío")
    if not row["ok_apellidos"]:
        errores.append("Apellido inválido o vacío")
    if not row["ok_dni"]:
        errores.append("DNI ausente o formato inválido (debe tener 7-8 dígitos)")
    if not row["ok_email_formato"]:
        errores.append("Email con formato inválido")
    if not row["ok_telefono_formato"]:
        errores.append("Teléfono con formato inválido")
    if not row["ok_contacto"]:
        errores.append("Sin medio de contacto válido (email o teléfono requerido)")
    if not row["ok_fecha_nac"]:
        errores.append("Fecha de nacimiento ausente o inválida")
    if not row["ok_fecha_alta"]:
        errores.append("Fecha de alta ausente o inválida")
    if not row["ok_fecha_baja"]:
        errores.append("Fecha de baja inválida")
    if not row["ok_baja_posterior"]:
        errores.append("Fecha de baja anterior a fecha de alta")
    if not row["ok_sexo"]:
        errores.append("Sexo inválido (valores aceptados: M, F, X)")
    return " | ".join(errores) if errores else ""

df_val["observaciones"] = df_val.apply(construir_observaciones, axis=1)

# Columnas de flags de validación (para separarlas al final)
COLS_FLAGS = [
    "ok_nombres", "ok_apellidos", "ok_dni", "ok_email_formato",
    "ok_telefono_formato", "ok_contacto", "ok_fecha_nac",
    "ok_fecha_alta", "ok_fecha_baja", "ok_baja_posterior", "ok_sexo",
    "fecha_nacimiento_dt", "fecha_alta_dt", "fecha_baja_dt",
]

# ── Separar: válidos vs. rechazados ───────────────────────────────────────────
mascara_validos = df_val["observaciones"] == ""

# Columnas originales del dataset
COLS_ORIGINALES = list(df_crudo.columns)

df_validos   = df_val[mascara_validos][COLS_ORIGINALES].copy().reset_index(drop=True)
df_rechazados = df_val[~mascara_validos][COLS_ORIGINALES + ["observaciones"]].copy().reset_index(drop=True)

print(f"Total registros      : {len(df_val)}")
print(f"Registros válidos    : {len(df_validos)}")
print(f"Registros rechazados : {len(df_rechazados)}")

In [ ]:
# ── Ver registros rechazados con sus motivos ───────────────────────────────────
print("=== REGISTROS RECHAZADOS ===")
df_rechazados[["id_cliente", "nombres", "apellidos", "observaciones"]]

In [ ]:
# ── Ver registros válidos ──────────────────────────────────────────────────────
print("=== REGISTROS VÁLIDOS ===")
df_validos

## 5. Normalización

Sobre el subconjunto de **registros válidos**, aplicamos normalización para garantizar  
consistencia en el formato de los valores antes de guardarlos en el sistema destino.

| Columna | Transformación |
|---------|---------------|
| `nombres` | `strip()` + `title()` |
| `apellidos` | `strip()` + `title()` |
| `email` | `strip()` + `lower()` |
| `sexo` | `strip()` + `upper()` |
| `numero_documento` | `strip()` |
| `nro_telefono` | `strip()` |
| Fechas | Convertir a `datetime` y estandarizar formato `YYYY-MM-DD` |

In [ ]:
df = df_validos.copy()

# ── Normalizar texto ───────────────────────────────────────────────────────────
df["nombres"]   = df["nombres"].str.strip().str.title()
df["apellidos"] = df["apellidos"].str.strip().str.title()
df["email"]     = df["email"].str.strip().str.lower()
df["sexo"]      = df["sexo"].str.strip().str.upper()

# ── Normalizar fechas ──────────────────────────────────────────────────────────
df["fecha_nacimiento"] = pd.to_datetime(df["fecha_nacimiento"]).dt.strftime("%Y-%m-%d")
df["fecha_alta"]       = pd.to_datetime(df["fecha_alta"]).dt.strftime("%Y-%m-%d")
df["fecha_baja"]       = pd.to_datetime(df["fecha_baja"], errors="coerce").dt.strftime("%Y-%m-%d")

# ── Normalizar id_cliente a entero ────────────────────────────────────────────
df["id_cliente"] = df["id_cliente"].astype(int)

print("=== DATASET NORMALIZADO ===")
df

## 6. Derivación de Columnas Calculadas

Creamos columnas nuevas a partir de los datos existentes para enriquecer el registro  
y facilitar análisis posteriores.

| Columna nueva | Descripción |
|---------------|-------------|
| `edad` | Años cumplidos a la fecha actual |
| `antiguedad_dias` | Días desde la `fecha_alta` |
| `estado` | `"Activo"` si `fecha_baja` es nula, `"Inactivo"` si tiene fecha_baja |
| `tiene_email` | Booleano: si registró email válido |
| `tiene_telefono` | Booleano: si registró teléfono válido |

In [ ]:
HOY = pd.Timestamp.today().normalize()

# Convertir fechas a datetime para cálculos
fecha_nac_dt  = pd.to_datetime(df["fecha_nacimiento"])
fecha_alta_dt = pd.to_datetime(df["fecha_alta"])
fecha_baja_dt = pd.to_datetime(df["fecha_baja"], errors="coerce")

# ── Edad en años completos ─────────────────────────────────────────────────────
df["edad"] = (
    (HOY - fecha_nac_dt).dt.days // 365
).astype(int)

# ── Antigüedad en días (desde la fecha_alta) ───────────────────────────────────
df["antiguedad_dias"] = (HOY - fecha_alta_dt).dt.days.astype(int)

# ── Estado del cliente ─────────────────────────────────────────────────────────
df["estado"] = np.where(fecha_baja_dt.isna(), "Activo", "Inactivo")

# ── Indicadores de contacto ────────────────────────────────────────────────────
df["tiene_email"]    = df["email"].notna().map({True: "Sí", False: "No"})
df["tiene_telefono"] = df["nro_telefono"].notna().map({True: "Sí", False: "No"})

print("=== COLUMNAS DERIVADAS ===")
df[["id_cliente", "nombres", "apellidos", "edad", "antiguedad_dias", "estado",
    "tiene_email", "tiene_telefono"]]

## 7. Resumen y Estadísticas

Generamos métricas de negocio sobre el dataset de clientes válidos  
para tener una visión global de la base de datos.

In [ ]:
# ── Distribución por sexo ──────────────────────────────────────────────────────
resumen_sexo = (
    df.groupby("sexo")
    .agg(
        cantidad       = ("id_cliente",     "count"),
        edad_promedio  = ("edad",           "mean"),
        edad_minima    = ("edad",           "min"),
        edad_maxima    = ("edad",           "max"),
    )
    .round(1)
    .reset_index()
)

print("=== DISTRIBUCIÓN POR SEXO ===")
resumen_sexo

In [ ]:
# ── Distribución por estado (activo / inactivo) ────────────────────────────────
resumen_estado = (
    df.groupby("estado")
    .agg(
        cantidad          = ("id_cliente",       "count"),
        antiguedad_prom   = ("antiguedad_dias",  "mean"),
    )
    .round(1)
    .reset_index()
)

print("=== DISTRIBUCIÓN POR ESTADO ===")
resumen_estado

In [ ]:
# ── Resumen de medios de contacto disponibles ──────────────────────────────────
total = len(df)
con_email    = df["email"].notna().sum()
con_telefono = df["nro_telefono"].notna().sum()
con_ambos    = (df["email"].notna() & df["nro_telefono"].notna()).sum()
solo_email   = (df["email"].notna() & df["nro_telefono"].isna()).sum()
solo_tel     = (df["email"].isna()  & df["nro_telefono"].notna()).sum()

print("=== MEDIOS DE CONTACTO ===")
print(f"Clientes con email         : {con_email}  ({con_email/total*100:.1f}%)")
print(f"Clientes con teléfono      : {con_telefono}  ({con_telefono/total*100:.1f}%)")
print(f"Clientes con ambos         : {con_ambos}  ({con_ambos/total*100:.1f}%)")
print(f"Solo email (sin teléfono)  : {solo_email}")
print(f"Solo teléfono (sin email)  : {solo_tel}")

In [ ]:
# ── Estadísticas de edad ───────────────────────────────────────────────────────
print("=== ESTADÍSTICAS DE EDAD ===")
print(df["edad"].describe().round(1))

print("\n=== RESUMEN GENERAL ===")
print(f"Total clientes válidos   : {len(df)}")
print(f"Total rechazados         : {len(df_rechazados)}")
print(f"Tasa de rechazo          : {len(df_rechazados)/len(df_crudo)*100:.1f}%")
print(f"Rango de antigüedad      : {df['antiguedad_dias'].min()} - {df['antiguedad_dias'].max()} días")

## 8. Guardado de Resultados

Guardamos tres archivos en `datos/output/`:

| Archivo | Contenido |
|---------|----------|
| `clientes_validos.csv` | Registros que pasaron todas las validaciones, normalizados y enriquecidos |
| `clientes_rechazados.csv` | Registros con errores, incluyendo la descripción de cada problema |
| `resumen_clientes.csv` | Métricas agregadas por sexo y estado |

In [ ]:
# ── 1. Clientes válidos ────────────────────────────────────────────────────────
ruta_validos = DIR_OUTPUT / "clientes_validos.csv"
df.to_csv(ruta_validos, index=False, encoding="utf-8")
print(f"Guardado: {ruta_validos}  ({len(df)} registros)")

# ── 2. Clientes rechazados ─────────────────────────────────────────────────────
ruta_rechazados = DIR_OUTPUT / "clientes_rechazados.csv"
df_rechazados.to_csv(ruta_rechazados, index=False, encoding="utf-8")
print(f"Guardado: {ruta_rechazados}  ({len(df_rechazados)} registros)")

# ── 3. Resumen por sexo ────────────────────────────────────────────────────────
ruta_resumen = DIR_OUTPUT / "resumen_clientes.csv"
resumen_sexo.to_csv(ruta_resumen, index=False, encoding="utf-8")
print(f"Guardado: {ruta_resumen}  ({len(resumen_sexo)} filas)")

print("\n✓ Todos los archivos guardados exitosamente en datos/output/")

## Resumen de la Clase

| Técnica | Función / herramienta | Para qué sirve |
|---------|----------------------|----------------|
| Cargar como string | `pd.read_csv(dtype=str)` | Evitar inferencia de tipos que oculte errores |
| Expresiones regulares | `re.compile()` + `.match()` | Validar formatos de email, DNI y teléfono |
| Validación de nulos | `.isna()` / `.notna()` | Detectar campos obligatorios ausentes |
| Parseo de fechas | `pd.to_datetime(errors='coerce')` | Detectar fechas inválidas (resultado: `NaT`) |
| Reglas de negocio | Condiciones booleanas combinadas | Aplicar lógica como "al menos un contacto" |
| Reporte de errores | `apply()` con función personalizada | Construir mensajes descriptivos por registro |
| Separar válidos/inválidos | Filtro con máscara booleana | Enrutar registros a destinos distintos |
| Normalización | `.str.strip().str.title()` | Consistencia de texto antes del almacenado |
| Columnas derivadas | Operaciones sobre `dt` y `np.where()` | Edad, antigüedad, estado del cliente |
| Agregaciones | `.groupby().agg()` | Métricas de la base de clientes |

> **Principio ETL:** La validación y el rechazo explícito de registros inválidos  
> es preferible a corregir datos sin certeza sobre su valor correcto.  
> Un registro rechazado con observaciones claras puede ser corregido en la fuente.